In [0]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
spark.sql("CREATE SCHEMA IF NOT EXISTS northstar.ml")

# Search EVERY experiment for your trained run (it's in the other notebook)
runs = mlflow.search_runs(
    search_all_experiments=True,
    filter_string="attributes.run_name = 'late_delivery_rf_balanced'",   # ← the balanced A2 run
    order_by=["start_time DESC"], max_results=1)

if runs.empty:
    raise ValueError("No matching run — check the run_name (try 'late_delivery_rf').")

run_id = runs.iloc[0]["run_id"]
model_uri = f"runs:/{run_id}/model"

mv = mlflow.register_model(model_uri=model_uri, name="northstar.ml.late_delivery")
print(f"✅ Registered version {mv.version}")

MlflowClient().set_registered_model_alias("northstar.ml.late_delivery", "champion", mv.version)
print("✅ alias 'champion' set")